# BTCDOM Data Verification

This notebook locates and verifies the **production-grade** BTCDOM datasets:
- Raw **Binance BTCDOM index**
- **Reconstructed BTCDOM index**

Rules:
- Ignore any paths under `archive/`, `archive_data/`, any `OldV*` folders, and any `*exercise*` scratchpads.
- Use only files under the curated data lake (or other non-archived production stores).


In [ ]:
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    # When executed via nbconvert, __file__ may not be defined; assume /notebooks under repo root
    BASE_DIR = Path.cwd().resolve().parents[0]

BASE_DIR

In [ ]:
# Helper: recursively search for BTCDOM-related files, excluding archives / scratchpads

EXCLUDE_DIR_KEYWORDS = {"archive", "archive_data", "oldv", "exercise"}

candidates = []
for root, dirs, files in os.walk(BASE_DIR):
    # Prune excluded directories
    rel_parts = Path(root).relative_to(BASE_DIR).parts
    if any(any(k in part.lower() for k in EXCLUDE_DIR_KEYWORDS) for part in rel_parts):
        continue

    for fname in files:
        fname_lower = fname.lower()
        if "btcdom" in fname_lower and fname_lower.endswith((".csv", ".parquet", ".db")):
            candidates.append(Path(root) / fname)

candidates = sorted(set(candidates))
print("Found BTCDOM-related candidate files (non-archived):")
for p in candidates:
    print(" -", p)

candidates

In [ ]:
# Select production files for Reconstructed and Binance BTCDOM

recon_path = None
binance_path = None

for p in candidates:
    name = p.name.lower()
    if "recon" in name or "reconstructed" in name:
        recon_path = p
    # Prefer explicit Binance BTCDOM market (e.g., BTCDOMUSDT.parquet)
    if "btcdomusdt" in name or "binance" in name:
        binance_path = p

# Fallback: if no explicit binance file, attempt to use btcdom_state.db as the raw source
if binance_path is None:
    state_db = BASE_DIR / "btcdom_state.db"
    if state_db.exists():
        binance_path = state_db

print("\nSelected production BTCDOM paths:")
print("  Reconstructed:", recon_path)
print("  Binance (raw):", binance_path)

recon_path, binance_path

In [ ]:
# Load reconstructed BTCDOM

if recon_path is None:
    raise FileNotFoundError("Could not locate reconstructed BTCDOM file (btcdom_reconstructed).")

if recon_path.suffix == ".csv":
    recon_df = pd.read_csv(recon_path, parse_dates=["date"])
elif recon_path.suffix == ".parquet":
    recon_df = pd.read_parquet(recon_path)
else:
    raise ValueError(f"Unsupported reconstructed BTCDOM format: {recon_path}")

# Identify reconstructed price column
recon_price_col = None
for col in recon_df.columns:
    cl = col.lower()
    if "reconstructed" in cl or "index" in cl:
        recon_price_col = col
        break
if recon_price_col is None:
    # Fallback: first numeric column after date
    numeric_cols = recon_df.select_dtypes(include=["number"]).columns.tolist()
    recon_price_col = numeric_cols[0]

print("\nReconstructed BTCDOM file:")
print(recon_path.resolve())
print("Price column:", recon_price_col)
print("\nReconstructed BTCDOM describe():")
print(recon_df[recon_price_col].describe())

In [ ]:
# Load raw Binance BTCDOM

if binance_path is None:
    raise FileNotFoundError("Could not locate raw Binance BTCDOM file (non-archived).")

print("\nBinance BTCDOM file:")
print(binance_path.resolve())

if binance_path.suffix == ".csv":
    binance_df = pd.read_csv(binance_path, parse_dates=["date"])
elif binance_path.suffix == ".parquet":
    binance_df = pd.read_parquet(binance_path)
elif binance_path.suffix == ".db":
    # Use sqlite3 to read from btcdom_state.db
    conn = sqlite3.connect(binance_path)
    try:
        tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist()
        if not tables:
            raise RuntimeError("No tables found in Binance BTCDOM state DB.")
        # Prefer table names that mention btcdom or index
        table = None
        for t in tables:
            tl = t.lower()
            if "btcdom" in tl or "index" in tl:
                table = t
                break
        if table is None:
            table = tables[0]
        binance_df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    finally:
        conn.close()
else:
    raise ValueError(f"Unsupported Binance BTCDOM format: {binance_path}")

# Heuristic to identify Binance price column
binance_price_col = None
for col in binance_df.columns:
    cl = col.lower()
    if "btcdom" in cl or "index" in cl:
        if np.issubdtype(binance_df[col].dtype, np.number):
            binance_price_col = col
            break
if binance_price_col is None:
    # Fallback: first numeric column that is not an obvious count/divisor
    numeric_cols = [c for c in binance_df.select_dtypes(include=["number"]).columns
                    if not any(k in c.lower() for k in ["divisor", "count"])]
    if not numeric_cols:
        numeric_cols = binance_df.select_dtypes(include=["number"]).columns.tolist()
    binance_price_col = numeric_cols[0]

print("Binance price column:", binance_price_col)
print("\nBinance BTCDOM describe():")
print(binance_df[binance_price_col].describe())

# Self-correction check: if max is implausibly high, warn
max_binance = binance_df[binance_price_col].max()
if max_binance > 4000:
    print("\n[WARNING] Binance BTCDOM max value > 4000 (", max_binance, ") - this file may still be legacy/corrupted.")

In [ ]:
# Visual overlay of Binance vs Reconstructed BTCDOM

# Standardize column names
recon_plot = recon_df[["date", recon_price_col]].rename(columns={"date": "date", recon_price_col: "recon_price"})

# Try to infer a date column for Binance
binance_date_col = None
for col in binance_df.columns:
    if col.lower() in {"date", "timestamp"}:
        binance_date_col = col
        break
if binance_date_col is None:
    # Fallback: assume first datetime-like column
    for col in binance_df.columns:
        if np.issubdtype(binance_df[col].dtype, np.datetime64):
            binance_date_col = col
            break
if binance_date_col is None:
    raise ValueError("Could not infer date column for Binance BTCDOM.")

# Ensure datetime
binance_df[binance_date_col] = pd.to_datetime(binance_df[binance_date_col])

binance_plot = binance_df[[binance_date_col, binance_price_col]].rename(
    columns={binance_date_col: "date", binance_price_col: "binance_price"}
)

# Inner join on overlapping date range
merged = pd.merge(recon_plot, binance_plot, on="date", how="inner").sort_values("date")

plt.figure(figsize=(10, 5))
plt.plot(merged["date"], merged["recon_price"], label="Reconstructed BTCDOM")
plt.plot(merged["date"], merged["binance_price"], label="Binance BTCDOM", alpha=0.7)
plt.legend()
plt.xlabel("Date")
plt.ylabel("Index Level")
plt.title("BTCDOM Verification: Binance vs Reconstructed")
plt.tight_layout()

out_path = BASE_DIR / "notebooks" / "btcdom_verification.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()

out_path